In [26]:
import os
from shutil import move

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from kagglehub import kagglehub
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from tqdm import tqdm
from skopt.space import Real, Integer, Categorical
from skopt import gp_minimize



In [27]:
if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available.")


CUDA is available!
Device name: NVIDIA GeForce RTX 4060 Laptop GPU


In [28]:
def create_dataloaders(train_dataset, test_dataset, batch_size, val_split=0.2):
    train_size = int((1 - val_split) * len(train_dataset))
    val_size = len(train_dataset) - train_size
    generator = torch.Generator().manual_seed(42)  # For reproducibility
    train_subset, val_subset = random_split(train_dataset, [train_size, val_size], generator=generator)

    if not hasattr(train_dataset, 'targets'):
        raise AttributeError("train_dataset must have 'targets' attribute for class balancing.")

    train_labels = [train_dataset.targets[i] for i in train_subset.indices]
    class_counts = np.bincount(train_labels)
    class_weights = 1. / np.sqrt(class_counts + 1e-6)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    sample_weights = [class_weights[label] for label in train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_subset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


In [29]:
def compute_dataset_stats(dataset):
    """
    Computes the mean and standard deviation of a grayscale dataset.

    Args:
        dataset (torch.utils.data.Dataset): Dataset for which to compute statistics.

    Returns:
        tuple: Mean and standard deviation for the single channel.
    """
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
    mean = 0.0
    std = 0.0
    total_samples = 0

    for images, _ in loader:
        batch_samples = images.size(0)  # Batch size
        images = images.view(batch_samples, -1)  # Flatten all pixels into one dimension
        mean += images.mean(1).sum(0)
        std += images.std(1).sum(0)
        total_samples += batch_samples

    mean /= total_samples
    std /= total_samples

    return mean.item(), std.item()


In [30]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs=100, log_interval=100, patience=5):
    """
    Trains a given model with streamlined logging and early stopping, while logging to TensorBoard.

    Args:
        model: The model to train.
        train_loader: DataLoader for the training dataset.
        val_loader: DataLoader for the validation dataset.
        optimizer: Optimizer for training (e.g., Adam, SGD).
        criterion: Loss function (e.g., CrossEntropyLoss).
        num_epochs: Number of training epochs.
        log_interval: Frequency (in batches) to log metrics during training.
        patience: Number of epochs to wait for validation loss improvement before stopping.
    """
    # Initialize TensorBoard writer
    writer = SummaryWriter(log_dir="../logs")

    # Set device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    best_val_loss = float('inf')
    patience_counter = 0
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # Move data to device
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Optimizer step
            optimizer.step()

            # Accumulate loss
            running_loss += loss.item()

            # Calculate training accuracy
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            # Log batch loss and accuracy to TensorBoard periodically
            if (batch_idx + 1) % log_interval == 0:
                avg_loss = running_loss / log_interval
                writer.add_scalar("Batch Training Loss", avg_loss, epoch * len(train_loader) + batch_idx)
                running_loss = 0.0

        # Training accuracy for the epoch
        train_accuracy = correct_train / total_train

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                # Validation accuracy
                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

        val_loss /= len(val_loader)
        val_accuracy = correct_val / total_val

        # Log epoch metrics to TensorBoard
        writer.add_scalar("Epoch Training Accuracy", train_accuracy, epoch)
        writer.add_scalar("Epoch Validation Accuracy", val_accuracy, epoch)
        writer.add_scalar("Epoch Validation Loss", val_loss, epoch)

        # Epoch summary
        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Acc: {train_accuracy:.4f}, Val Acc: {val_accuracy:.4f}, Val Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

    writer.close()


In [31]:
def evaluate_model(model, data_loader):
    """
    Evaluates the model's performance on a given dataset.

    Args:
        model: The trained model to evaluate.
        data_loader: DataLoader for the dataset to evaluate.

    Returns:
        float: Accuracy of the model on the dataset.
    """
    # Set the device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Move model to the device
    model = model.to(device)

    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            # Move data to the same device as model
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy


In [32]:
def objective(params):
    # Unpack hyperparameters
    learning_rate, batch_size, optimizer_type, dropout_rate, weight_decay, patience = params
    batch_size = int(batch_size)
    patience = int(patience)

    # Log the parameters being tested
    print(
        f"Training with params: lr={learning_rate}, batch_size={batch_size}, optimizer={optimizer_type}, "
        f"dropout_rate={dropout_rate}, weight_decay={weight_decay}, patience={patience}"
    )

    # Set the device (use GPU if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Update data loaders
    train_loader, val_loader, _ = create_dataloaders(train_dataset, test_dataset, batch_size)

    # Define model
    model = CNN(dropout_rate=dropout_rate).to(device)  # Pass dropout_rate to CNN

    # Define loss function
    criterion = nn.CrossEntropyLoss()

    # Define optimizer dynamically based on `optimizer_type`
    optimizer = getattr(optim, optimizer_type)(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )

    # Train model using the provided `train_model` method
    train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        num_epochs=100,
        log_interval=100,
        patience=patience,
    )

    # Evaluate model after training
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = correct / total
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    return -val_accuracy  # Return negative accuracy for minimization


In [33]:
class CNN(nn.Module):
    def __init__(self, input_size=(48, 48), num_classes=8, dropout_rate=0.3):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)  # Batch Norm
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Compute fully connected input size dynamically
        fc_input_size = 128 * (input_size[0] // 8) * (input_size[1] // 8)

        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(fc_input_size, 256)
        self.fc2 = nn.Linear(256, num_classes)

        self._init_weights()

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)


In [34]:
# Generalized permanent dataset path
home_dir = os.path.expanduser("~")  # Gets the user's home directory
permanent_path = os.path.join(home_dir, "datasets")

# Download dataset if it doesn't exist
if not os.path.exists(permanent_path):
    print("Downloading dataset...")
    temp_path = kagglehub.dataset_download("subhaditya/fer2013plus")  # Temporary download path
    print("Moving dataset to permanent location...")
    move(temp_path, permanent_path)
    print("Dataset moved to:", permanent_path)
else:
    print("Dataset already exists at:", permanent_path)

# Identify train and test directories dynamically
train_dir = os.path.join(permanent_path, "fer2013plus", "fer2013", "train")
test_dir = os.path.join(permanent_path, "fer2013plus", "fer2013", "test")

Dataset already exists at: /home/omarhammad/datasets


In [35]:
# Paths for augmented datasets
augmented_train_dir = os.path.join(permanent_path, "fer2013plus/fer2013/train_augmented")
augmented_test_dir = os.path.join(permanent_path, "fer2013plus/fer2013/test_augmented")

# Define augmentations for train and processing for test

# Compute stats for grayscale normalization
train_stats = compute_dataset_stats(datasets.ImageFolder(train_dir, transform=transforms.ToTensor()))
mean, std = train_stats
print(f"Computed Mean: {mean}, Std: {std}")

train_augment_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Ensure grayscale
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomCrop(48, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Brightness/Contrast
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),  # Use single-channel stats
    transforms.RandomErasing(p=0.5),  # Add random erasing
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),  # Use single-channel stats
])


# Check if the augmented datasets exist
if not os.path.exists(augmented_train_dir) or not os.path.exists(augmented_test_dir):
    print("Augmented datasets not found. Creating augmented datasets...")

    # Use GPU for processing if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Create and save the augmented train dataset
    print("Processing train dataset...")
    for label in os.listdir(train_dir):
        label_dir = os.path.join(train_dir, label)
        save_dir = os.path.join(augmented_train_dir, label)
        os.makedirs(save_dir, exist_ok=True)
        for img_name in tqdm(os.listdir(label_dir), desc=f"Processing {label}"):
            img_path = os.path.join(label_dir, img_name)
            img = Image.open(img_path).convert("L")  # Convert to grayscale (1 channel)

            # Apply augmentations to the PIL Image
            for i in range(5):  # Generate 5 augmented images per original
                augmented_img = train_augment_transform(img)  # Apply transforms directly to the PIL Image
                augmented_img_pil = transforms.ToPILImage()(augmented_img)  # Convert tensor back to PIL Image for saving
                augmented_img_pil.save(os.path.join(save_dir, f"{os.path.splitext(img_name)[0]}_aug_{i}.png"))

    # Process and save the test dataset
    print("Processing test dataset...")
    for label in os.listdir(test_dir):
        label_dir = os.path.join(test_dir, label)
        save_dir = os.path.join(augmented_test_dir, label)
        os.makedirs(save_dir, exist_ok=True)
        for img_name in tqdm(os.listdir(label_dir), desc=f"Processing {label}"):
            img_path = os.path.join(label_dir, img_name)
            img = Image.open(img_path).convert("L")  # Convert to grayscale (1 channel)

            # Apply test transforms directly
            processed_img = test_transform(img)
            processed_img_pil = transforms.ToPILImage()(processed_img)  # Convert tensor back to PIL Image for saving
            processed_img_pil.save(os.path.join(save_dir, img_name))
else:
    print("Augmented datasets already exist. Skipping processing.")

# Reload processed datasets with normalization applied during loading
train_dataset = datasets.ImageFolder(augmented_train_dir, transform=train_augment_transform)
test_dataset = datasets.ImageFolder(augmented_test_dir, transform=test_transform)


Computed Mean: 0.5076549053192139, Std: 0.21201835572719574
Augmented datasets already exist. Skipping processing.


In [36]:
# Dataset sizes
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 141930
Test dataset size: 7099


In [37]:
search_space = [
    Real(1e-6, 1e-2, name='learning_rate', prior='log-uniform'),
    Integer(16, 64, name='batch_size'),
    Categorical(['Adam', 'SGD', 'RMSprop', 'AdamW'], name='optimizer'),
    Real(0.1, 0.5, name='dropout_rate'),
    Real(1e-5, 1e-2, name='weight_decay', prior='log-uniform'),
    Integer(3, 10, name='patience')  # Early stopping patience
]


In [38]:

result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=10,
    random_state=42
)

# Define the parameter names in the same order as in the search space
param_names = ['learning_rate', 'batch_size', 'optimizer', 'dropout_rate', 'weight_decay', 'patience']

# Print each parameter with its name
print("Best parameters:")
for name, value in zip(param_names, result.x):
    print(f"{name} = {value}")

# Print the best validation accuracy
print(f"Best validation accuracy: {-result.fun:.4f}")


Training with params: lr=0.001535224694197351, batch_size=25, optimizer=AdamW, dropout_rate=0.33874006317859484, weight_decay=0.00021751953118777657, patience=4
Epoch [1/100], Train Acc: 0.2521, Val Acc: 0.3812, Val Loss: 1.6784
Epoch [2/100], Train Acc: 0.2730, Val Acc: 0.4135, Val Loss: 1.6433
Epoch [3/100], Train Acc: 0.2973, Val Acc: 0.4481, Val Loss: 1.5417
Epoch [4/100], Train Acc: 0.3464, Val Acc: 0.4825, Val Loss: 1.4663
Epoch [5/100], Train Acc: 0.3729, Val Acc: 0.4969, Val Loss: 1.4327
Epoch [6/100], Train Acc: 0.3887, Val Acc: 0.5017, Val Loss: 1.3965
Epoch [7/100], Train Acc: 0.4054, Val Acc: 0.5158, Val Loss: 1.3495
Epoch [8/100], Train Acc: 0.4104, Val Acc: 0.5155, Val Loss: 1.3536
Epoch [9/100], Train Acc: 0.4185, Val Acc: 0.5070, Val Loss: 1.3783
Epoch [10/100], Train Acc: 0.4267, Val Acc: 0.5347, Val Loss: 1.3059
Epoch [11/100], Train Acc: 0.4347, Val Acc: 0.5423, Val Loss: 1.2847
Epoch [12/100], Train Acc: 0.4397, Val Acc: 0.5371, Val Loss: 1.2942
Epoch [13/100], Trai

KeyboardInterrupt: 

In [ ]:
# Extract best parameters from tuning results
best_learning_rate = result.x[0]
best_batch_size = int(result.x[1])
best_optimizer_type = result.x[2]
best_dropout_rate = result.x[3]
best_weight_decay = result.x[4]
best_patience = int(result.x[5])

# Update data loaders with the best batch size
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, test_dataset, best_batch_size)

# Define the model, criterion, and optimizer
best_cnn_model = CNN(dropout_rate=best_dropout_rate)
criterion = nn.CrossEntropyLoss()

# Use the best optimizer type with weight decay
optimizer = getattr(torch.optim, best_optimizer_type)(
    best_cnn_model.parameters(),
    lr=best_learning_rate,
    weight_decay=best_weight_decay
)

# Train the model with early stopping
train_model(
    model=best_cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    patience=best_patience
)

In [ ]:
train_accuracy = evaluate_model(best_cnn_model, train_loader)
print(f"Train Accuracy: {train_accuracy:.4f}")

In [ ]:
test_accuracy = evaluate_model(best_cnn_model, test_loader)
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Save the model's state_dict
torch.save(best_cnn_model.state_dict(), 'best_cnn_model.pth')